## Inter-Agent Communication (A2A)
### Overview
The Agent2Agent (A2A) protocol is an open standard designed to enable communication and collaboration between different AI agent frameworks. It ensures interoperability, allowing AI agents developed with technologies like LangGraph, CrewAI, or GoogleADK to work together regardless of their origin or framework differences.

The A2A protocols provides a structured approach for agent interactions, built upon several core concepts. The foundational pillars of A2A invlude Core Actors, Agent Card, Agent Discovery, Communication and Tasks, Interaction mechanism, and Security, all of which will be reviewed in detail.

__Core Actors__: A2A involves thre emain entities:
- User: initiates requests for agent assistance.
- A2A Client (Client Agent): An application or AI agent that acts on the user's behalf to request actions or information.
- A2A Server (Remote Agent): An AI agent or system that provides an HTTP endpoint to process client requests and return results. The remote agent operates as an "opaque" system, meaning the client does not need to understand its internal operational details.

__Agent Card__: An agent's digital identity is defined by its Agent Card, usually a JSON file. This file contains key information for client interaction and automatic discovery, including the agent's identity, endpoint URL, and version. It also details supported capabilities like streaming or push notifications, specific skills, default input/output modes, and authentication requirements. Below is an example of an Agent Card for a WeatherBot.

```json
{
    "name": "WeatherBot",
    "description": "Provides accurate weather forecasts and historical data.",
    "url": "http://weather-service.example.com/a2a",
    "version": "1.0.0",
    "capabilities": {
        "streaming": true,
        "pushNotifications": false,
        "stateTransitionHistory": true
    },
    "authentication": {
        "schemes": [
            "apiKey"
        ]
    },
    "defaultInputModes": [
        "text"
    ],
    "defaultOutputModes": [
        "text"
    ],
    "skills": [
        {
            "id": "get_current_weather",
            "name": "Get Current Weather",
            "description": "Retrieve real-time weather for any location.",
            "inputModes": [
                "text"
            ],
            "outputModes": [
                "text"
            ],
            "examples": [
                "What's the weather in Paris?",
                "Current conditions in Tokyo"
            ],
            "tags": [
                "weather",
                "current",
                "real-time"
            ]
        },
        {
            "id": "get_forecast",
            "name": "Get Forecast",
            "description": "Get 5-day weather predictions.",
            "inputModes": [
                "text"
            ],
            "outputModes": [
                "text"
            ],
            "examples": [
                "5-day forecast for New York",
                "Will it rain in London this weekend?"
            ],
            "tags": [
                "weather",
                "forecast",
                "prediction"
            ]
        }
    ]
}
```

__Agent Discovery__: it allows agents to find Agent Cards, which describe the capabilities of available A2A servers. Several strategies exist for this process:
- Well-known URI: agents host their Agent Card at a standardized path (e.g., /well-known/agent.json). This approach offesrs broad, often automated, accessibility for public or domain-specific use.

- Curated Registries: These provide a centralized catalog where Agent Cards are published and can be queried based on specific criteria. This is well-suited for enterprise environments needing centralized management and access control.

- Direct Configuration: Agent Card information is embedded or privately shared. This method is appropriate for closely coupled or private systems where dynamic iscovery isn't crucial.

Regardless of the chosen method, it's important to secure Agent Card endpoints. This can be achieved through access control, mutual TLS (mTLS), or network restrictions, especially if the card contains sensitive (though non-secret) information.

__Communications and Tasks__: in the A2A framework, communication is structured around asynchronous tasks, which represent the fundamental units of work for long-running processes. Each task is assigned a unique identifier and moves through a series of states - such as submitted, working, or completed - a design that supports parallel processing in complex operations. Communication between agents occurs through a Message.

This communication contains attributes, which are key-value metadata describing the message (like its priority or creation time), and one or more parts, which carry the actual content being delivered, such as plain text, files, or structured JSON data. The tangible outputs generated by an agent during a task are called artifacts. Like messages, artifacts are also composed of one or more parts and can be streamed incrementally as results become available. All communication within the A2A framework is conducted over HTTP(S) using teh JSON-RPC 2.0 protocol for payloads. To maintain continuity across multiple interactions, a server-generated `contextID` is used to group related tasks and preserve context.

__Interaction Mechanisms__:  Request/Response (Polling) Server-Sent Events (SSE): A2A provides multiple interaction methods to suit a variety of AI application needs, each with a distinct mechanism:
- Synchronous Request/Response: For qhick, immediate operations. In this model, the client sends a request and actively waits for the server to process it and return a complete response in a single, synchronous exhange.

- Asynchronous Polling: Suited for tasks that take longer to process. This client sends a request, and the server immediately acknowledges it with a "working" status and a task ID. The client is then free to perform other actions and can periodically poll the server by sending new requests to check the status of the task until it is marked as "completed" or "failed".

- Streaming Updates (Server-Events - SSE): Ideal for receiving real-time, incremental results. This method establishes a persistent, one-way connection from the server to the client. It allows the remote agent to continuously push updates, such as status changes or partial results, without the client needing to make multiple requests.

- Push Notifications (Webhooks): Designed for very long-running or resource-intensive tasks where maintaining a constant connection or frequent polling is inefficient. This client can register a webhook URL, and the server will send an asynchronous notification (a "push") to that URL when the task's status changes significantly (e.g., upon completion).

The Agent Card specifies whether an agent supports streaming or push notification capabilities. Furthermore, A2A is modality-agnostic, meaning it can facilitate these interaction patterns not just for text, but also for other data types like audio and video, enablish rich, multimodal AI applications. Both streaming and push notification capabilities are specified within the Agent Card.

```json
# Synchronous Request Example 
{
    "jsonrpc": "2.0",
    "id": "1",
    "method": "sendTask",
    "params": {
        "id": "task-001",
        "sessionId": "session-001",
        "message": {
            "role": "user",
            "parts": [
                {
                    "type": "text",
                    "text": "What is the exchange rate from USD to EUR?"
                }
            ]
        },
        "acceptedOutputModes": [
            "text/plain"
        ],
        "historyLength": 5
    }
}
```

The synchronous request uses the `sendTask` method, where the client asks for an expects a single, complete answer to its query. In contrast, the streaming request uses the `sendTaskSubscribe` method to establish a persistent connection, allowing teh agent to send back multiple, incremental updates or partial results over time.

```json
# Streaming Request Example 
{
    "jsonrpc": "2.0",
    "id": "2",
    "method": "sendTaskSubscribe",
    "params": {
        "id": "task-002",
        "sessionId": "session-001",
        "message": {
            "role": "user",
            "parts": [
                {
                    "type": "text",
                    "text": "What's the exchange rate for JPY to GBP today?"
                }
            ]
        },
        "acceptedOutputModes": [
            "text/plain"
        ],
        "historyLength": 5
    }
}
```

### A2A vs MCP
A2A is a protocol that complements the MCP protocol. While MCP focuses on structuring context for agents and their interaction with external data and tools, A2A facilitates coordination and communication among agents, enabling task delegation and collaboration.

### Practical Applications and Use Cases
A2A is used for building sophisticated AI solutions across diverse domains, enabling modularity, scalability, and enhanced intelligence.
- Multi-Framework Collaboration: A2A's primary use case is enabling independent AI agents, regardless of their underlying frameworks (e.g., Langchain) to communicate and collaborate. This is fundamental for building complex multi-agent systems where different agents specialize in different aspects of a problem.

- Automated Workflow Orchestration: In enterprise settings, A2A can facilitate complex workflows by enabling agents to delegate and coordinate tasks. For instance, and agent might handle initial data collection, then delegate to another agent for analysis, and finally to a third for report generation, all communicating via the A2A protocol.

- Dynamic Information Retrieval: Agents can communicate to retrieve and exchange real-time information. A primary agent might request live market data from a specialized "data fetching agent", which then uses external APIs to gather the information and send it back.

### Hands-On Code Example:
Better explained here: https://github.com/google-a2a/a2a-samples/tree/main/samples